# Databricks/PySpark for Data Analysts/Engineers — Module 13: MLlib -- Model Evaluation and Tuning

In Module 12 we built our first models and evaluated them with a single metric (RMSE,
AUC) on a single data split. This module shows how to do it more rigorously: why one
train/test split isn't enough, how to automatically search many hyperparameter
configurations, and how to pick the best model without manual guesswork.

## Table of contents
1. [Train/test/validation -- why one split isn't enough](#sec1)
2. [Evaluators in depth: regression and classification metrics](#sec2)
3. [ParamGridBuilder: a hyperparameter grid](#sec3)
4. [CrossValidator: cross-validation](#sec4)
5. [Summary and exercises](#sec5)


We start (or grab the already-running) `SparkSession` -- the entry point to everything in Spark. In Databricks, `spark` is already available automatically in every notebook; the same code also works locally.

In [ ]:
from pyspark.sql import SparkSession

# Extra JVM options (--add-opens) are needed when running Spark locally on Java 17+ and
# using pandas UDFs / Apache Arrow (Module 8) -- newer JDKs block, by default, the
# reflective off-heap memory access that Arrow needs. In Databricks you don't need to
# set this by hand -- the platform handles it for you.
_jvm_options = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("spark_course")
    .config("spark.driver.extraJavaOptions", _jvm_options)
    .config("spark.executor.extraJavaOptions", _jvm_options)
    .getOrCreate()
)
spark


Let's load our shared dataset -- online store data (`customers`, `products`, `employees`, `orders`, `order_items`), the exact same schema as the SQL course, but this time as Spark DataFrames.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- customers -----------------------------------------------------------
n_customers = 300
cities = ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segments = ["New", "Standard", "Premium"]
segment_probs = [0.35, 0.45, 0.20]

customers_pd = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "city": rng.choice(cities, size=n_customers, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segments, size=n_customers, p=segment_probs),
    "days_since_signup": rng.integers(1, 900, size=n_customers),
})

# --- products ------------------------------------------------------------
n_products = 40
categories = ["Electronics", "Clothing", "Home", "Sports", "Books"]
avg_prices = {"Electronics": 1200, "Clothing": 150, "Home": 250, "Sports": 300, "Books": 50}

product_categories = rng.choice(categories, size=n_products)
products_pd = pd.DataFrame({
    "product_id": np.arange(1, n_products + 1),
    "category": product_categories,
    "price": [
        round(max(5.0, rng.normal(avg_prices[c], avg_prices[c] * 0.3)), 2)
        for c in product_categories
    ],
})

# --- employees ----------------------------------------------------------
n_employees = 12
regions = ["North", "South", "East", "West"]
employees_pd = pd.DataFrame({
    "employee_id": np.arange(1, n_employees + 1),
    "region": rng.choice(regions, size=n_employees),
    "department": rng.choice(["Sales", "Support"], size=n_employees, p=[0.6, 0.4]),
})

# --- orders ------------------------------------------------------------
n_orders = 3000
order_customer_id = rng.choice(customers_pd["customer_id"], size=n_orders)
order_employee_id = rng.choice(employees_pd["employee_id"], size=n_orders)

segment_map = customers_pd.set_index("customer_id")["segment"]
order_segment = pd.Series(order_customer_id).map(segment_map).values

# cancellation probability depends on customer segment (deliberately built in)
cancel_prob = np.where(
    order_segment == "New", 0.22,
    np.where(order_segment == "Standard", 0.10, 0.04),
)
cancelled = rng.binomial(1, cancel_prob)
order_status = np.where(
    cancelled == 1, "Cancelled",
    rng.choice(["Placed", "Shipped", "Delivered"], size=n_orders, p=[0.2, 0.3, 0.5]),
)

base_date = pd.Timestamp("2024-01-01")
day_offset = rng.integers(0, 730, size=n_orders)

orders_pd = pd.DataFrame({
    "order_id": np.arange(1, n_orders + 1),
    "customer_id": order_customer_id,
    "employee_id": order_employee_id,
    "order_date": [base_date + pd.Timedelta(days=int(d)) for d in day_offset],
    "status": order_status,
})

# --- order_items ------------------------------------------------------
item_rows = []
item_id = 1
product_price = products_pd.set_index("product_id")["price"]

for order_id in orders_pd["order_id"]:
    n_items = rng.integers(1, 5)
    chosen_products = rng.choice(products_pd["product_id"], size=n_items, replace=False)
    for product_id in chosen_products:
        quantity = int(rng.integers(1, 4))
        discount = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        item_rows.append({
            "item_id": item_id,
            "order_id": int(order_id),
            "product_id": int(product_id),
            "quantity": quantity,
            "unit_price": float(product_price[product_id]),
            "discount": float(discount),
        })
        item_id += 1

order_items_pd = pd.DataFrame(item_rows)

# --- convert to Spark DataFrames ---------------------------------------
customers = spark.createDataFrame(customers_pd)
products = spark.createDataFrame(products_pd)
employees = spark.createDataFrame(employees_pd)
orders = spark.createDataFrame(orders_pd)
order_items = spark.createDataFrame(order_items_pd)

customers.show(5)


> ⚡ **spark.sql.shuffle.partitions -- the default 200 is too many for small data**
>
> `CrossValidator` in section 4 trains dozens of models back to back -- on our SMALL dataset, the default 200 partitions on every shuffle (Module 9) is pure task-scheduling overhead, with no benefit from parallelism. We lower this value for the rest of this notebook -- in real work with large data you'd leave the default, or tune it to your cluster's size.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "8")


In [ ]:
from pyspark.sql import functions as F

order_features = (
    orders.join(customers, "customer_id")
    .join(order_items, "order_id")
    .groupBy("order_id", "customer_id", "segment", "status")
    .agg(
        F.count("*").alias("item_count"),
        F.avg("unit_price").alias("avg_price"),
        F.sum(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))).alias("value"),
    )
    .withColumn("is_cancelled", (F.col("status") == "Cancelled").cast("double"))
)
order_features.show(5)


<a id="sec1"></a>
## 1. Train/test/validation -- why one split isn't enough

In Module 12 we split data into ONLY two parts: train and test. That works for a simple
demonstration, but has a pitfall: if you use the TEST set repeatedly to compare
different model configurations and pick the best one, in practice you start "fitting"
to that particular test set -- similar to repeatedly correcting your answers while
looking at the same test.

In [ ]:
# A three-way split: train (for learning), validation (for choosing hyperparameters),
# test (a single, final evaluation -- touched only once, at the very end)
train, validation, test = order_features.randomSplit([0.6, 0.2, 0.2], seed=42)
print(f"Train: {train.count()}, validation: {validation.count()}, test: {test.count()}")


> ⚠️ **You touch the test set ONLY ONCE, at the very end**
>
> A rule that's easy to break by accident: the VALIDATION set is for comparing configurations and picking the best one (you can use it repeatedly). The TEST set is ONLY for one, final evaluation of the chosen model -- once you start adjusting the model based on it, it stops being a trustworthy, independent quality measure. `CrossValidator` (section 4) automates the train/validation split for you -- you still always keep a separate test set on the side.

<a id="sec2"></a>
## 2. Evaluators in depth: regression and classification metrics

`RegressionEvaluator` and `BinaryClassificationEvaluator` (Module 12) support MANY
metrics, not just the default one -- changed via `metricName` or `setMetricName()`.

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

assembler = VectorAssembler(inputCols=["item_count", "avg_price"], outputCol="features")
reg_model = LinearRegression(featuresCol="features", labelCol="value")

trained_assembled = assembler.transform(train)
model = reg_model.fit(trained_assembled)
predictions = model.transform(assembler.transform(test))

for metric in ["rmse", "mse", "mae", "r2"]:
    evaluator = RegressionEvaluator(labelCol="value", predictionCol="prediction", metricName=metric)
    print(f"{metric.upper()}: {evaluator.evaluate(predictions):.3f}")


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

log_model = LogisticRegression(featuresCol="features", labelCol="is_cancelled")
model_log = log_model.fit(assembler.transform(train))
log_predictions = model_log.transform(assembler.transform(test))

for metric in ["areaUnderROC", "areaUnderPR"]:
    evaluator = BinaryClassificationEvaluator(
        labelCol="is_cancelled", rawPredictionCol="rawPrediction", metricName=metric
    )
    print(f"{metric}: {evaluator.evaluate(log_predictions):.3f}")

for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="is_cancelled", predictionCol="prediction", metricName=metric
    )
    print(f"{metric}: {evaluator.evaluate(log_predictions):.3f}")


> ⚡ **accuracy can be misleading with imbalanced classes**
>
> In our data, the large majority of orders are NOT cancelled -- a model that ALWAYS predicts 'not cancelled' will achieve a high `accuracy`, despite being useless. We covered this exact problem with logistic regression in Module 13 of the Statistics course -- that's why `areaUnderROC`/`areaUnderPR` (insensitive to class imbalance) are more trustworthy metrics here than `accuracy` alone.

<a id="sec3"></a>
## 3. ParamGridBuilder: a hyperparameter grid

Instead of MANUALLY trying different hyperparameter values (e.g. `regParam` --
regularization strength), `ParamGridBuilder` builds a GRID of every combination to test
automatically.

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder

grid = (
    ParamGridBuilder()
    .addGrid(reg_model.regParam, [0.0, 0.1, 0.5, 1.0])
    .addGrid(reg_model.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
print(f"Number of combinations to test: {len(grid)}")


> 🧱 **regParam and elasticNetParam -- a reminder from the Statistics course**
>
> `regParam` controls the STRENGTH of regularization (0 = none, larger values = stronger penalty on large coefficients, counteracting overfitting). `elasticNetParam` mixes two types of regularization: 0 is pure Ridge (L2), 1 is pure Lasso (L1) -- the same concepts you'd encounter with regularized regression in `sklearn`/`statsmodels`.

<a id="sec4"></a>
## 4. CrossValidator: cross-validation

`CrossValidator` combines `ParamGridBuilder` with **k-fold cross-validation**: for EVERY
hyperparameter combination, it splits the training data into `k` parts, trains `k`
times (each time on a different combination of parts as train/validation), averages the
result, and finally picks the COMBINATION with the best average result.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator

pipeline = Pipeline(stages=[assembler, reg_model])

pipeline_grid = (
    ParamGridBuilder()
    .addGrid(reg_model.regParam, [0.0, 0.1, 0.5, 1.0])
    .addGrid(reg_model.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

cv_evaluator = RegressionEvaluator(labelCol="value", predictionCol="prediction", metricName="rmse")

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=pipeline_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    seed=42,
)

cv_model = cv.fit(train.union(validation.select(train.columns)))
print(f"Average RMSE for each combination: {[round(m, 2) for m in cv_model.avgMetrics]}")


In [ ]:
# The best model, chosen automatically -- evaluated on the test set, untouched so far
best_model = cv_model.bestModel
best_predictions = best_model.transform(test)

best_rmse = cv_evaluator.evaluate(best_predictions)
print(f"Best model's RMSE on the test set: {best_rmse:.2f}")

best_regression = best_model.stages[-1]
print(f"Best regParam: {best_regression.getRegParam()}")
print(f"Best elasticNetParam: {best_regression.getElasticNetParam()}")


> ⚠️ **CrossValidator is computationally expensive -- number of combinations times numFolds**
>
> Our grid had 4 x 3 = 12 combinations, with `numFolds=3` -- that's 36 full model trainings. On large data and bigger grids, this cost grows fast. `TrainValidationSplit` (a sibling class to `CrossValidator`, not covered in detail here) does only ONE train/validation split instead of `k`-fold -- faster, but a less stable result. Choosing between them is a trade-off between compute time and evaluation reliability.

<a id="sec5"></a>
## 5. Summary and exercises

In this module we covered:
- the train/validation/test split -- and why we touch the test set only once,
- the full set of `RegressionEvaluator` metrics (rmse/mse/mae/r2) and `BinaryClassificationEvaluator`/`MulticlassClassificationEvaluator` metrics (AUC, accuracy, F1, precision, recall),
- `ParamGridBuilder` -- building a hyperparameter grid to test automatically,
- `CrossValidator` -- k-fold cross-validation and automatically choosing the best configuration.

### 📝 Exercise 1: A three-way split for products

Split `products` into train/validation/test with a 60/20/20 ratio (`seed=1`) and print the row count of each part.

<details>
<summary>Show solution</summary>

```python
tr, val, te = products.randomSplit([0.6, 0.2, 0.2], seed=1)
print(f"Train: {tr.count()}, validation: {val.count()}, test: {te.count()}")
```

</details>

### 📝 Exercise 2: All regression metrics at once

For the regression model from section 2, print ALL four metrics (rmse, mse, mae, r2) in a single `for` loop.

<details>
<summary>Show solution</summary>

```python
for metric in ["rmse", "mse", "mae", "r2"]:
    ev = RegressionEvaluator(labelCol="value", predictionCol="prediction", metricName=metric)
    print(f"{metric}: {ev.evaluate(predictions):.3f}")
```

</details>

### 📝 Exercise 3: A simple hyperparameter grid

Build a `ParamGridBuilder` testing ONLY `regParam` at values `[0.0, 0.3, 0.6]` (no `elasticNetParam`) and print the number of combinations.

<details>
<summary>Show solution</summary>

```python
from pyspark.ml.tuning import ParamGridBuilder

grid = ParamGridBuilder().addGrid(reg_model.regParam, [0.0, 0.3, 0.6]).build()
print(f"Number of combinations: {len(grid)}")
```

</details>

### 📝 Exercise 4: CrossValidator for logistic regression

Build a `CrossValidator` (numFolds=3) for a `Pipeline` with logistic regression (feature: `item_count`, label: `is_cancelled`), testing `regParam` over `[0.0, 0.1, 0.5]`, and print the best `regParam`.

<details>
<summary>Show solution</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

simple_assembler = VectorAssembler(inputCols=["item_count"], outputCol="features")
simple_log_model = LogisticRegression(featuresCol="features", labelCol="is_cancelled")
simple_pipeline = Pipeline(stages=[simple_assembler, simple_log_model])

grid = ParamGridBuilder().addGrid(simple_log_model.regParam, [0.0, 0.1, 0.5]).build()
evaluator = BinaryClassificationEvaluator(
    labelCol="is_cancelled", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=simple_pipeline, estimatorParamMaps=grid, evaluator=evaluator, numFolds=3, seed=1
)
cv_model = cv.fit(order_features)
print(f"Best regParam: {cv_model.bestModel.stages[-1].getRegParam()}")
```

</details>

> 🔥 **Challenge: a full comparison of a default model vs. a tuned model**
>
> Build TWO linear regression models predicting `value`: (1) with default hyperparameters, (2) chosen by `CrossValidator` over a `regParam` x `elasticNetParam` grid (as in section 4). Compare the RMSE of BOTH models on the same, shared test set -- did tuning actually improve the result?

<details>
<summary>Show solution</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

tr, val, te = order_features.randomSplit([0.6, 0.2, 0.2], seed=42)

assembler = VectorAssembler(inputCols=["item_count", "avg_price"], outputCol="features")
model = LinearRegression(featuresCol="features", labelCol="value")
pipeline = Pipeline(stages=[assembler, model])
evaluator = RegressionEvaluator(labelCol="value", predictionCol="prediction", metricName="rmse")

# Model 1: default hyperparameters
default_model = pipeline.fit(tr)
default_rmse = evaluator.evaluate(default_model.transform(te))

# Model 2: chosen by CrossValidator
grid = (
    ParamGridBuilder()
    .addGrid(model.regParam, [0.0, 0.1, 0.5, 1.0])
    .addGrid(model.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=grid, evaluator=evaluator, numFolds=3, seed=42)
cv_model = cv.fit(tr.union(val.select(tr.columns)))
tuned_rmse = evaluator.evaluate(cv_model.bestModel.transform(te))

print(f"RMSE (default hyperparameters): {default_rmse:.2f}")
print(f"RMSE (after CrossValidator tuning): {tuned_rmse:.2f}")
```

On a feature set this simple and small, the difference may be minor -- hyperparameter tuning gives the biggest benefit with more features and a higher risk of overfitting, where regularization actually has something to do.

</details>

## What's next?

In **Module 14** we'll cover working in the Databricks Workspace itself: notebooks,
compute clusters, scheduling jobs (Jobs/Workflows), widgets, and `dbutils` -- the
practical side of the platform, beyond the Spark code itself.